# 01 - Explore Bronze Layer

Interactive exploration of raw data landing in the Bronze Delta table.
Use this notebook to verify data is flowing and inspect raw message structure.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("explore-bronze")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
# Read Bronze table
bronze = spark.read.format("delta").load("s3a://cryptostream-dev-bronze/raw_events/")
print(f"Total rows: {bronze.count():,}")
bronze.printSchema()

In [ ]:
# Row count by topic and date
(
    bronze
    .groupBy("kafka_topic", "dt")
    .count()
    .orderBy("dt", "kafka_topic")
    .show(20)
)

In [ ]:
# Sample raw messages
(
    bronze
    .select("kafka_topic", "message_key", "message_value", "ingested_at")
    .orderBy(F.col("ingested_at").desc())
    .limit(5)
    .show(truncate=80)
)

In [ ]:
# Check ingestion latency (last 100 records)
from pyspark.sql.types import TimestampType
import json

recent = (
    bronze
    .filter(F.col("kafka_topic") == "crypto.trades.v1")
    .orderBy(F.col("ingested_at").desc())
    .limit(100)
    .withColumn("payload", F.from_json("message_value", "E LONG, _received_at LONG"))
    .withColumn("event_time", (F.col("payload.E") / 1000).cast("timestamp"))
    .withColumn("received_time", (F.col("payload._received_at") / 1000).cast("timestamp"))
    .withColumn("ws_latency_ms", F.col("payload._received_at") - F.col("payload.E"))
    .select("event_time", "received_time", "ingested_at", "ws_latency_ms")
)

recent.describe("ws_latency_ms").show()

In [ ]:
# Delta table history
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, "s3a://cryptostream-dev-bronze/raw_events/")
dt.history(10).select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

In [ ]:
spark.stop()